In [1]:
from sarma.ingestion.loader import load_pdf
from sarma.ingestion.splitter import split_documents
from sarma.vectorstore.vectorstore import create_vector_store
from sarma.retriever import create_retriever
from sarma.rag.rag import create_rag_chain
from sarma.llm import llm

C:\Users\rost8\anaconda3\envs\sarma\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6295.85it/s]


In [6]:
documents = load_pdf("../data/raw/RB209 Arable crops.pdf")
chunks = split_documents(documents)
db = create_vector_store(chunks)
retriever = create_retriever(db)
chain = create_rag_chain(retriever)

In [3]:
response = chain.invoke("What is the best Nitrogen value for wheat?")
print(response.content)

The best nitrogen value for wheat, based on the provided examples and recommendations, is **120 kg N/ha** for a **medium soil** (as stated in Example 4.3). This recommendation is derived from Table 4.15 (page 28) and assumes a soil nitrogen supply (SNS) index of 4. However, the "best" value may vary depending on factors like soil type, previous crop history, rainfall, and SNS index (e.g., Example 4.5 uses an SNS index of 2 for a deep clay soil in a high rainfall area). For precise adjustments, consult the specific tables (e.g., Table 4.15 or 4.21) and consider site-specific conditions. 

**Answer:** The recommended nitrogen value for wheat in a medium soil is **120 kg N/ha**, but adjustments may be needed based on soil type, crop history, and SNS index.


In [5]:
results = db.similarity_search_with_score(
"What is the best Nitrogen value for wheat?",
k=10
)

for doc, score in results:
    print('SCORE:', score)
    print(doc.page_content[:200])
    print("="*50)

SCORE: 0.35917893052101135
32
Cereals
Wheat, spring sown – nitrogen
Table 4.18 Nitrogen for spring sown wheat
SNS Index
0 1 2 3 4 5 6
kg N/ha
Light sand soils 160 130 100 70 40 0–40 0
All other mineral soils 210a 180 150 120 70
SCORE: 0.38683944940567017
29
Cereals
Barley, winter sown – nitrogen
Table 4.16 Nitrogen for winter-sown barley
SNS Index
0 1 2 3 4 5 6
kg N/ha
Feed barley
Light sand soils 170 140 110 80 60 0–40 0
Shallow soils 220a 190 150 12
SCORE: 0.3951227068901062
28
Cereals
Wheat and triticale, sown up to the end of January – nitrogen
Table 4.15 Nitrogen for wheat and triticale (sown up to the end of January)
SNS Index
0 1 2 3 4 5 6
kg N/ha
Light sand soils 18
SCORE: 0.4096190929412842
fertiliser application rates should be adjusted down or up by 25 kg N/ha per 
0.5% difference in grain protein (30 kg N/ha per 0.1% difference in grain %N).
To convert grain %protein to grain %N, divi
SCORE: 0.42135030031204224
and lowest, and take an average of the remaining three years.
W

In [8]:
print(db._collection.count())

113


In [13]:
items = db._collection.get(
    limit=10,
    include=["metadatas", "documents"]
)

for meta in items["metadatas"]:
    print(meta)

{'page_label': '1', 'title': 'Sentinel-2 Products Specification Document', 'creationdate': '2021-10-05T15:46:31+02:00', 'keywords': '1', 'page': 0, 'source': '..\\data\\Sentinel-2-product-specifications-document-V14-9.pdf', 'author': 'Annamaria', 'moddate': '2021-10-05T16:01:07+02:00', 'creator': 'Microsoft® Word 2010', 'total_pages': 552, 'producer': 'Microsoft® Word 2010'}
{'author': 'Annamaria', 'page_label': '2', 'creator': 'Microsoft® Word 2010', 'creationdate': '2021-10-05T15:46:31+02:00', 'producer': 'Microsoft® Word 2010', 'total_pages': 552, 'source': '..\\data\\Sentinel-2-product-specifications-document-V14-9.pdf', 'keywords': '1', 'page': 1, 'title': 'Sentinel-2 Products Specification Document', 'moddate': '2021-10-05T16:01:07+02:00'}
{'total_pages': 552, 'author': 'Annamaria', 'creator': 'Microsoft® Word 2010', 'page': 1, 'page_label': '2', 'producer': 'Microsoft® Word 2010', 'keywords': '1', 'title': 'Sentinel-2 Products Specification Document', 'moddate': '2021-10-05T16:0

In [7]:
response = chain.invoke(
    "What nitrogen rate is recommended for spring sown wheat?"
)

print(response.content)

The recommended nitrogen rate for spring sown wheat is determined based on the crop's total nitrogen requirement and development stage, with specific application timings outlined in the documents. For crops drilled before March, nitrogen should be applied at early stem extension (GS30–31) but not before early April or after early May. The exact rate depends on the total nitrogen requirement, with splits as follows:

- **Less than 100 kg N/ha**: Apply as a single dressing by early stem extension (GS30–31).  
- **Between 100 and 200 kg N/ha**: Split the application, with half during late tillering (mid-February/early March) and half at GS30–31.  
- **200 kg N/ha or more**: Apply three splits (40% during late tillering, 40% at GS30–31, and 20% at GS32).  

For spring sown wheat, the rate is adjusted based on expected yield and soil nitrogen supply. For example, winter wheat recommendations in the documents (e.g., 120 kg N/ha for a medium soil) may serve as a reference, but spring sown whe